# Prophet

Создай `df` в этом же ядре. В таблице должно быть два столбца: дата и значение ряда. Имена столбцов любые, на дату нужна одна строка. Для остатков возьми `REP_DT` и `OST_VAL` по одной валюте.

Прогноз строится по дням до той же даты следующего месяца включительно. Prophet учитывает недельную сезонность. Если пакетов ещё нет, выполни `%pip install "pandas>=2" "prophet>=1.1.5" matplotlib` в отдельной ячейке.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from prophet import Prophet

WEEKLY_SEASONALITY = True
YEARLY_SEASONALITY = False
CHANGEPOINT_PRIOR_SCALE = 0.05
SEASONALITY_PRIOR_SCALE = 10.0
SHOW_STAN_CONSOLE = False
MONTH = pd.DateOffset(months=1)

if df.shape[1] != 2:
    raise ValueError("Нужны два столбца: дата и значение ряда.")
date_col, value_col = df.columns
dates = pd.DatetimeIndex(pd.to_datetime(df.iloc[:, 0], format="mixed"), name=date_col)
dates = dates.tz_localize(None).normalize()
values = pd.to_numeric(df.iloc[:, 1]).astype(float)
y = pd.Series(values.to_numpy(), index=dates, name=value_col).sort_index()

if y.notna().sum() < 2 or dates.hasnans or dates.has_duplicates:
    raise ValueError("Нужны уникальные даты и минимум два непустых значения.")
if y.abs().eq(float("inf")).any():
    raise ValueError("Значения ряда должны быть конечными числами или пропусками.")

y = y.loc[:y.last_valid_index()].asfreq("D")

## MAE и MAPE на истории

Даты делятся по порядку на `train` (60%), `valid` (20%) и `test` (20%). Первая половина `train` служит начальной историей. Для каждого месячного блока Prophet обучается на всех известных значениях к дате выпуска. Факты блока используются при обучении со следующей даты выпуска. Последний блок в каждой выборке может быть короче месяца.

Пропуски остаются на дневной сетке, модель обучается по имеющимся фактам. MAE считается по всем известным фактам в единицах ряда, MAPE — по ненулевым фактам в процентах.

Prophet масштабирует остатки по диапазону на обучающих данных. Это сохраняет небольшие колебания ряда при большом уровне остатков. Если обучение остановилось, включи `SHOW_STAN_CONSOLE = True` в настройках и повтори запуск, чтобы увидеть вывод CmdStan.

In [ ]:
def fit(history, uncertainty_samples=0):
    observed = history.dropna()
    model = Prophet(
        weekly_seasonality=WEEKLY_SEASONALITY,
        yearly_seasonality=YEARLY_SEASONALITY,
        daily_seasonality=False,
        changepoint_prior_scale=CHANGEPOINT_PRIOR_SCALE,
        seasonality_prior_scale=SEASONALITY_PRIOR_SCALE,
        interval_width=0.95,
        scaling="minmax",
        uncertainty_samples=uncertainty_samples,
    )
    training = pd.DataFrame({"ds": observed.index, "y": observed.to_numpy()})
    try:
        return model.fit(training, seed=42, show_console=SHOW_STAN_CONSOLE)
    except RuntimeError as error:
        raise RuntimeError(
            f"Сбой оптимизации Prophet для выпуска {history.index[-1].date()}, "
            f"известных значений: {len(observed)}. "
            "Для подробностей включи SHOW_STAN_CONSOLE в настройках."
        ) from error


def predict(model, dates):
    return model.predict(pd.DataFrame({"ds": dates})).set_index("ds")


def month_days(origin):
    return ((origin + MONTH).date() - origin.date()).days


train_end, valid_end = int(len(y) * 0.6), int(len(y) * 0.8)
warmup = train_end // 2
if y.iloc[:warmup].count() < 2:
    raise ValueError("Недостаточно истории перед первым прогнозом на train.")

splits = {"train": (warmup, train_end), "valid": (train_end, valid_end),
          "test": (valid_end, len(y))}
blocks = []
for split, (start, stop) in splits.items():
    while start < stop:
        origin = y.index[start - 1]
        end = min(start + month_days(origin), stop)
        actual = y.iloc[start:end]
        model = fit(y.iloc[:start])
        predicted = predict(model, actual.index)["yhat"]
        blocks.append(pd.DataFrame({"split": split, "origin": origin,
                                    "actual": actual, "forecast": predicted}, index=actual.index))
        start = end

backtest = pd.concat(blocks)
error = (backtest["actual"] - backtest["forecast"]).abs()
denominator = backtest["actual"].abs().replace(0, float("nan"))
metrics = (backtest.assign(MAE=error, MAPE_pct=100 * error / denominator)
           .groupby("split", sort=False)[["MAE", "MAPE_pct"]].mean())
display(metrics.round(3))

## Прогноз на следующий месяц

После проверки на истории Prophet обучается на всех известных значениях. Таблица `forecast` содержит дневной прогноз на месяц от последнего факта и 95%-й интервал прогноза.

In [ ]:
origin = y.index[-1]
future_dates = pd.date_range(origin + pd.offsets.Day(), origin + MONTH, freq="D", name=date_col)
model = fit(y, uncertainty_samples=1000)
prediction = predict(model, future_dates)
forecast = prediction[["yhat", "yhat_lower", "yhat_upper"]].rename(columns={
    "yhat": "forecast", "yhat_lower": "lower_95", "yhat_upper": "upper_95",
}).rename_axis(date_col)
display(forecast)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 11), layout="constrained")
for ax, (split, rows) in zip(axes, backtest.groupby("split", sort=False)):
    ax.plot(rows.index, rows["actual"], color="C0", label="Факт")
    ax.plot(rows.index, rows["forecast"], color="C1", label="Прогноз")
    ax.set_title(split, loc="left")
    ax.legend()

ax = axes[-1]
recent = y.tail(90)
ax.plot(recent.index, recent, color="C0", label="Факт")
ax.plot(forecast.index, forecast["forecast"], color="C1", label="Прогноз")
ax.fill_between(forecast.index, forecast["lower_95"], forecast["upper_95"],
                color="C1", alpha=0.18, label="95%-й интервал")
ax.set_title("Следующий месяц", loc="left")
ax.legend()
plt.show()

## Параметры Prophet

| Параметр | Что задаёт |
| --- | --- |
| `WEEKLY_SEASONALITY` | Повторяющиеся эффекты дней недели. |
| `YEARLY_SEASONALITY` | Сезонность в течение года. Для её оценки нужна история за несколько лет. |
| `CHANGEPOINT_PRIOR_SCALE` | Гибкость тренда в точках изменения. Большее значение позволяет быстрее менять наклон. |
| `SEASONALITY_PRIOR_SCALE` | Сила регуляризации сезонных компонентов. При меньших значениях сезонность сглаживается сильнее. |

Сравни настройки по MAE на `valid`, затем оцени выбранный вариант на `test`. Столбец `trend` в `prediction` показывает тренд, а столбец `weekly` — недельный компонент, если он включён. [Документация Prophet](https://meta.github.io/prophet/docs/diagnostics.html#hyperparameter-tuning)